# WCT only operation optimization

This notebook is used to prototype the operation optimization of the WCT only 
cooling system. It should aim at finding the optimal operation of the WCT, that is,
providing the required cooling power at the lowest possible cost (electricity 
consumption and water consumption)

In this notebook two problems are analyzed:
- Un-restricted water consumption with a single water source (WCT-1S-U)
- Two water sources where the cheaper one is restricted in volume (WCT-2S-R)

## Restricted problem (WCT-2S-R)

![Draft 2 sources - restricted](./assets/wct-2s-r.png)


In [1]:
import os
MR = "/home/patomareao/MATLAB/R2024b"
os.environ['LD_LIBRARY_PATH'] = f"{MR}/runtime/glnxa64:\
{MR}/bin/glnxa64:\
{MR}/sys/os/glnxa64:\
{MR}/extern/bin/glnxa64:\
/lib/x86_64-linux-gnu:"


In [2]:
from pathlib import Path
from dataclasses import asdict
import numpy as np
import pandas as pd
# from IPython.display import SVG
from loguru import logger
import pygmo as pg
import hjson

import combined_cooler_model
from solhycool_modeling import OperationPoint

from solhycool_modeling import EnvironmentVariables
from solhycool_optimization.problems.horizon import WctRestrictedProblem as Problem
from solhycool_optimization.utils.evaluation import optimize

# Visualization packages
from solhycool_optimization.visualization import plot_obj_scape_comp_1d
# from solhycool_visualization.operation import plot_hydraulic_distribution
# from solhycool_visualization.diagrams import WascopStateVisualizer
from solhycool_optimization.visualization import visualize_solutions_distribution
from solhycool_visualization.operation import plot_results

logger.disable("phd_visualizations")

data_path: Path = Path("../../data")
env_path: Path = data_path / "datasets/environment_data_20220101_20241231.h5"
base_output_path: Path = Path("../results")
diagram_path: Path = Path("/workspaces/SOLhycool/data/assets/base_diagram.svg")

cc_model = combined_cooler_model.initialize()
np.set_printoptions(precision=2)

%load_ext autoreload
%autoreload 2


In [3]:
# Load environment into EnvironmentVariables for the episode
df_env = pd.read_hdf(env_path)
df_env


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td,...,Q_kW,Tv_C,water_price_morocco_eur_m3,water_price_morocco_alternative_eur_m3,water_price_egypt_eur_m3,water_price_egypt_alternative_eur_m3,Vavail_m3,deltaV_m3_h,Vavail_l,deltaV_l_h
2022-01-01 00:00:00+00:00,0.0,0.0,0.0,0.0,10.7,90.0,0.0,-144.0,6.7,9.1,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.083485,-0.916515,19083.485034,-916.514966
2022-01-01 01:00:00+00:00,0.0,0.0,0.0,0.0,11.6,82.0,0.0,-163.4,0.5,8.7,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.020231,-0.979769,19020.230685,-979.769315
2022-01-01 02:00:00+00:00,0.0,0.0,0.0,0.0,11.3,82.0,0.0,-124.4,0.5,8.4,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.002447,-0.997553,19002.447115,-997.552885
2022-01-01 03:00:00+00:00,0.0,0.0,0.0,0.0,11.2,82.0,0.0,-105.4,0.3,8.3,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,18.965136,-1.034864,18965.135822,-1034.864178
2022-01-01 04:00:00+00:00,0.0,0.0,0.0,0.0,11.0,82.0,0.0,-93.6,0.2,8.1,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,18.968172,-1.031828,18968.171943,-1031.828057
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 18:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.047302,-0.952698,19047.301795,-952.698205
2024-12-31 19:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.090256,-0.909744,19090.255607,-909.744393
2024-12-31 20:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.048952,-0.951048,19048.952026,-951.047974
2024-12-31 21:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.098820,-0.901180,19098.819632,-901.180368


### Evaluate from available environment
#### Full day

In [4]:
selected_date_str: str = "20220615" 
df_ = df_env.loc[selected_date_str]

output_path = base_output_path / selected_date_str
if not output_path.exists():
    output_path.mkdir(parents=True)


In [5]:
env_vars = EnvironmentVariables.from_dataframe(df_)
env_vars.Q = env_vars.Q/2 # With only one component we can only get to half of the nominal power

# for var, value in asdict(env_vars).items():
#     print(var, value)


In [6]:
problem = Problem(env_vars=env_vars)
display(problem.get_bounds())
prob = pg.problem(problem)
x = pg.random_decision_vector(prob)
problem.fitness(x)


(array([5.22, 5.22, 5.22, 5.22, 5.22, 5.22, 5.22, 5.22, 5.22, 5.22, 5.22,
        5.22, 5.22, 5.22, 5.22, 5.22, 5.22, 5.22, 5.22, 5.22, 5.22, 5.22,
        5.22, 5.22, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
        0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
        0.  , 0.  , 0.  , 0.  ]),
 array([24.15, 24.15, 24.15, 24.15, 24.15, 24.15, 24.15, 24.15, 24.15,
        24.15, 24.15, 24.15, 24.15, 24.15, 24.15, 24.15, 24.15, 24.15,
        24.15, 24.15, 24.15, 24.15, 24.15, 24.15, 93.42, 93.42, 93.42,
        93.42, 93.42, 93.42, 93.42, 93.42, 93.42, 93.42, 93.42, 93.42,
        93.42, 93.42, 93.42, 93.42, 93.42, 93.42, 93.42, 93.42, 93.42,
        93.42, 93.42, 93.42]))

135.1876427741772


[135.1876427741772,
 3.0388569903509435,
 -3.687126094739696,
 1.6974935782935283,
 0.037602156794463326,
 4.010759025337052,
 -3.9490104477328956,
 11.059172901298346,
 5.626195689883318,
 7.730511938949842,
 -2.1351677450596185,
 0.3944613041399023,
 -4.071320925545528,
 2.3858084522675433,
 -4.162899921152324,
 7.685171601583491,
 -2.2019085562614222,
 1.5562489089793736,
 -4.13660067389155,
 2.7975048342640676,
 -3.9834874893420107,
 0.5993092240223419,
 -4.031524334903864,
 0.6842991584307008,
 -4.010853390725671,
 0.43836929012988435,
 -4.163760798006898,
 1.7203864440318384,
 -4.144150507263404,
 1.3180993716271772,
 -2.7152800717687313,
 1.2953231112488908,
 -4.16121659097017,
 7.429496727423789,
 -4.159517149171954,
 3.985053041581981,
 -1.9569413925968036,
 0.7592088423806871,
 -4.03621428502246,
 5.47836015365089,
 -3.0169997547654788,
 5.681269259971366,
 8.244709096810325,
 1.8416350443928096,
 -1.6026886977571735,
 4.785873378037291,
 6.9976434306352076,
 1.74807549926714

In [7]:
problem.evaluate(x, return_dataframe=True)


,Tamb,HR,mv,Tv,qc,Tc_in,Tc_out,Tcond,Qc_released,Qc_absorbed,...,Vavail_s1,Je,Jw,J,Je_c,Je_dc,Je_wct,Jw_wct,Jw_s1,Jw_s2
0,20.5,49.0,297.777268,35.0,19.975599,24.687126,33.312874,35.0,200.002084,200.067278,...,18.862183,0.607012,5.029577,5.636590,0.519941,0.0,0.087072,5.029577,5.029577,0.0
1,19.7,48.0,297.777268,35.0,10.718649,20.962398,37.037602,35.0,200.002084,200.067278,...,18.633247,0.456313,6.651477,7.107790,0.114035,0.0,0.342278,6.651477,6.651477,0.0
2,19.2,49.0,297.777268,35.0,21.266961,24.949010,33.050990,35.0,200.002084,200.067278,...,18.501440,0.633923,3.829521,4.463444,0.593081,0.0,0.040842,3.829521,3.829521,0.0
3,18.7,48.0,297.777268,35.0,6.322545,15.373804,42.626196,35.0,200.002084,200.067278,...,18.316292,0.090622,5.379251,5.469873,0.045515,0.0,0.045107,5.379251,5.379251,0.0
4,18.4,49.0,297.777268,35.0,14.689634,23.135168,34.864832,35.0,200.002084,200.067278,...,18.228416,0.260716,2.553140,2.813856,0.240793,0.0,0.019923,2.553140,2.553140,0.0
5,18.3,47.0,148.888634,35.0,20.320995,28.689111,32.928679,35.0,100.001042,100.021107,...,17.932911,0.999005,8.585573,9.584577,0.546548,0.0,0.452457,8.585573,8.585573,0.0
6,18.8,48.0,148.888634,35.0,13.597872,26.501385,32.837100,35.0,100.001042,100.028713,...,17.786122,0.296059,4.264803,4.560862,0.203896,0.0,0.092163,4.264803,4.264803,0.0
7,19.7,45.0,148.888634,35.0,7.429362,23.201909,34.798091,35.0,100.001042,100.033639,...,17.750345,0.071307,1.039469,1.110776,0.053163,0.0,0.018144,1.039469,1.039469,0.0
8,20.9,43.0,148.888634,35.0,16.463051,27.630333,32.863399,35.0,100.001042,100.024736,...,17.506154,0.609312,7.094686,7.703998,0.280401,0.0,0.328910,7.094686,7.094686,0.0
9,22.2,37.0,148.888634,35.0,10.724757,24.983487,33.016513,35.0,100.001042,100.033639,...,17.362868,0.253493,4.163026,4.416518,0.101105,0.0,0.152387,4.163026,4.163026,0.0


In [14]:
# algo = pg.algorithm(pg.cstrs_self_adaptive(iters=20, algo=pg.de(gen=10)))
algo = pg.algorithm(
    # pg.mbh(
        pg.gaco(gen=100)
    # )
)
algo.set_verbosity(1)
pop = pg.population(prob, 100)
pop = algo.evolve(pop)



   Gen:        Fevals:          Best:        Kernel:        Oracle:            dx:            dp:
      1              0        91.0684             63              0         997.59        23.4353
      2            100        86.4451             63              0        820.292        18.9523
      3            200        86.4451             63              0        857.535        14.0271
      4            300        86.4451             63              0         1038.5        11.1253
      5            400        78.3565             63              0        695.477        14.9981
      6            500        75.8133             63              0        642.805        12.6499
      7            600        67.3274             63              0        862.441        13.8642
      8            700        60.2831             63              0        436.115        11.7653
The command was cancelled. 


TypeError: cannot unpack non-iterable bool object

In [60]:
env_vars = EnvironmentVariables.from_dataframe(df_)
env_vars.Q = env_vars.Q/2 # With only one component we can only get to half of the nominal power
display(env_vars)

problem = Problem(env_vars=env_vars)
op_pts_list, _, _, fitness_list = optimize(
    problem, extra_outputs=True, max_iter=30, use_mbh=True, 
    algo_id="ipopt", n_trials = 1, log_verbosity = 1,
)
best_idx = np.argmin(fitness_list[:,0])
    
df_results = pd.DataFrame(
    [asdict(op_pt) for op_pt in op_pts_list[best_idx]], 
    index=df_.index)
df_results.to_csv(output_path / "wct_r_preliminary_results.csv")
logger.info(f"Results saved in {output_path}")
df_results


EnvironmentVariables(HR=array([49., 48., 49., 48., 49., 47., 48., 45., 43., 37., 37., 37., 36.,
       37., 37., 37., 37., 36., 36., 37., 39., 41., 46., 50.]), Tamb=array([20.5, 19.7, 19.2, 18.7, 18.4, 18.3, 18.8, 19.7, 20.9, 22.2, 23.5,
       24.6, 25.6, 26.3, 26.8, 27. , 27. , 26.7, 26.1, 25.1, 24.1, 23.1,
       22. , 21. ]), Tv=array([35., 35., 35., 35., 35., 35., 35., 35., 35., 35., 35., 35., 35.,
       35., 35., 35., 35., 35., 35., 35., 35., 35., 35., 35.]), Q=array([100., 100., 100., 100., 100.,  50.,  50.,  50.,  50.,  50.,  50.,
        50.,  50.,  50.,  50.,  50.,  50.,  50.,  50.,  50., 100., 100.,
       100., 100.]), Pe=array([0.18, 0.17, 0.17, 0.18, 0.18, 0.18, 0.18, 0.17, 0.15, 0.15, 0.15,
       0.16, 0.15, 0.14, 0.15, 0.15, 0.14, 0.16, 0.18, 0.18, 0.18, 0.15,
       0.2 , 0.17]), Pw=array([0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03,
       0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03,
       0.03, 0.03]), Pw_s1=array([0.03, 0

2025-03-01 17:11:37.857 | INFO     | solhycool_optimization.utils.evaluation:optimize:54 - Iteration 1 out of 1


The command was cancelled. 


TypeError: cannot unpack non-iterable bool object

In [ ]:
visualize_solutions_distribution(fitness_list[:, 0])


In [ ]:
if use_mbh:
    log = algo_list[-1].extract(pg.mbh).get_log()
else:
    log = algo_list[-1].extract(getattr(pg, algo_id)).get_log()

algo_log = pd.DataFrame(log, columns=["objevals","objval","violated","viol. norm", "valid"])

fig = plot_obj_scape_comp_1d([algo_log["objval"].values, algo_log["viol. norm"].values], 
                             algo_ids=["Fitness", "Inequality constraint"],
                             title_text="<b>Fitness evolution</b><br>Interior Point Optimization (IPOPT)")
[
    fig.add_hline(y=c_tol, 
                  line=dict(color="green", width=2, dash="dash"), 
                  name=f"Constraint tolerance {idx}", showlegend=True)
    for idx, c_tol in enumerate(problem.c_tol)
]
fig


In [ ]:
df_results = pd.read_csv(output_path / "wct_r_preliminary_results.csv", index_col=0, parse_dates=True)
plot_config = hjson.load(open(data_path / "plot_config.hjson"))

plot_results(plot_config, df_results)
